# Data preparation and validation for Russian-adapted CEFR simplification experiments

In [ ]:
import pandas as pd

# CEFR-labelled sentence dataset
cefr = pd.read_csv("CEFR_level_sentences.csv", sep=';', encoding='cp1251')
cefr

,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",A1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",A1
2,На каждой двери красные плакаты и красные фона...,A1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",A1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,A1
...,...,...
7167,Тем важнее становилась в эти годы семья для те...,C1
7168,"Учёные считают, что наибольшую угрозу может пр...",C1
7169,Не так давно футболист Александр Кержаков встр...,C1
7170,Этот столбик-указатель назвали гномоном. Гномо...,C1


In [ ]:
# source–target simplification dataset (complex vs simplified sentences)
source_target = pd.read_csv("source_target_sentences.csv", sep=';', encoding='cp1251')
source_target

,source,cos_sim,target,level
0,Ну что ж.,1.000000,Ну что ж.,b2
1,"— Ах, только не мужем, с простою усмешкой сказ...",0.863004,"— Ах, только не мужем, — с простою усмешкой ск...",b1
2,"Мне скоро шестьдесят, я старик, одинокий, ничт...",0.653318,"Ничего во мне нет хорошего, кроме этой любви к...",b2
3,Да!,0.093739,Обстановочка-то какова!,a2_b1
4,"Я вполне уверен, что вы раскаялись и раскаивае...",0.621839,"Я уверен, что вы раскаялись и будете содейство...",b1
...,...,...,...,...
24227,У нас в феврале масленица,0.960889,У нас в феврале масленица.,b1_c1
24228,— Разводиться мне или нет?,0.962737,— Разводиться мне или нет?,b1
24229,"Парусиновую куртку Петр Алексеевич сбросил, ру...",0.938603,"Парусиновую куртку Пётр Алексеевич сбросил, р...",b1_b2
24230,Ай!,1.000000,Ай!,b2


In [ ]:
# target CEFR-levelled vocabulary dataset
vocabulary = pd.read_csv("vocabulary.csv", sep=';', encoding='cp1251')
vocabulary

,base,level
0,а,A1
1,август,A1
2,автобус,A1
3,автомобиль,A1
4,автор,A1
...,...,...
8942,эскадра,C2
8943,эскадрилья,C2
8944,эшелон,C2
8945,языковый,C2


## Adapt the vocabulary list

In [ ]:
vocabulary.rename(columns={'word': 'base'}, inplace=True)
vocabulary.to_csv('new_vocabulary.csv', index=False)

In [ ]:
new_vocabulary = pd.read_csv("new_vocabulary.csv")
new_vocabulary

,base,level
0,а,A1
1,август,A1
2,автобус,A1
3,автомобиль,A1
4,автор,A1
...,...,...
8942,эскадра,C2
8943,эскадрилья,C2
8944,эшелон,C2
8945,языковый,C2


## Vocabulary level statistics

In [ ]:
import pandas as pd

VOCAB_PATH = "vocabulary.csv"

CEFR2ABC = {
    "A1": "A",
    "A2": "A",
    "B1": "B",
    "B2": "B",
    "C1": "C",
    "C2": "C",
}

vocab = pd.read_csv(VOCAB_PATH, encoding="windows-1251", sep=";")

vocab["word"] = vocab["word"].astype(str).str.strip()
vocab["level"] = vocab["level"].astype(str).str.strip().str.upper()

vocab["abc"] = vocab["level"].map(CEFR2ABC)

vocab = vocab.dropna(subset=["abc"])

print(vocab.head())
print(vocab["level"].value_counts())
print(vocab["abc"].value_counts())

         word level abc
0           а    A1   A
1      август    A1   A
2     автобус    A1   A
3  автомобиль    A1   A
4       автор    A1   A

level
B2    3422
B1    1562
C1    1393
A2    1056
A1     932
C2     582
Name: count, dtype: int64

abc
B    4984
A    1988
C    1975
Name: count, dtype: int64


In [ ]:
A_words = vocab[vocab["abc"] == "A"]["word"].drop_duplicates().tolist()
B_words = vocab[vocab["abc"] == "B"]["word"].drop_duplicates().tolist()
C_words = vocab[vocab["abc"] == "C"]["word"].drop_duplicates().tolist()

print("A:", len(A_words))
print("B:", len(B_words))
print("C:", len(C_words))

print("\nПримеры A:", A_words[:10])
print("Примеры B:", B_words[:10])
print("Примеры C:", C_words[:10])

A: 1988
B: 4984
C: 1975

Примеры A: ['а', 'август', 'автобус', 'автомобиль', 'автор', 'адрес', 'адресат', 'азбука', 'азиатский', 'азия']
Примеры B: ['абонемент', 'авиакомпания', 'автозаправочная станция', 'автоинспектор', 'автоматически', 'автоматический', 'автоответчик', 'авторизоваться', 'агентство', 'адвокат']
Примеры C: ['авиационный', 'авиация', 'авторитетный', 'авторский', 'авторское право', 'авторство', 'адмирал', 'акт', 'активы', 'акционер']


# CEFR level sentences statistics

In [ ]:
import pandas as pd

df = pd.read_csv("CEFR_level_sentences.csv", sep=";", encoding="cp1251")
df = df[["fragment", "textbook-assigned cefr level"]].dropna()
df["textbook-assigned cefr level"] = df["textbook-assigned cefr level"].astype(str).str.strip().str.upper()

def bucket(lbl: str) -> str:
    if lbl in ("A1", "A2"):
        return "A"
    if lbl in ("B1", "B2"):
        return "B"
    if lbl in ("C1", "C2"):
        return "C"
    return "UNK"

df["bucket"] = df["textbook-assigned cefr level"].apply(bucket)
print(df["bucket"].value_counts())
print(df[df["bucket"] == "UNK"]["textbook-assigned cefr level"].value_counts().head(20))

bucket
B    4091
A    1899
C    1182
Name: count, dtype: int64
Series([], Name: count, dtype: int64)


## Validate data_utils

In [ ]:
data = load_cefr_data("A", mode="reward")
print("Train chosen example:\n", data["train"]["chosen"][0])
print("\nTrain rejected example:\n", data["train"]["rejected"][0])

Train chosen example:
 У нас не очень большая, но удобная квартира: прихожая, кухня, спальня, гостиная, детская, ванная и туалет.

Train rejected example:
 Учёный без общения – это не учёный. У нас в институте существует такой порядок: человек написал статью – тут же выносит её на обсуждение.


In [ ]:
sents = get_complicated_sentence("source_target_sentences.csv")
print(type(sents), len(sents), sents[0][:80])

<class 'list'> 24232 Ну что ж.
